# ETS MARL — Episode Viewer

Inspect **one** episode at the finest grain available. Use this notebook to:

- Load a configuration (and optionally a trained checkpoint, e.g. the final episode of a long run).
- Roll out a single deterministic episode and capture every per-year log + per-company snapshot.
- Switch between a **market perspective** (cap / TNAC / MSR / clearing prices / volumes / defaults) and a **company perspective** (budgets, transactions, investments, compliance, rewards).
- Drill into a single year (`INSPECT_YEAR`) or a single company across the whole episode (`INSPECT_AGENT`).

Everything below is plain pandas / matplotlib; nothing is hidden in helper modules. Tweak the controls in the next cell, run all, and use the section headers as a table of contents.


## 1. Controls

Set the inputs for the rollout. The notebook is fully self-contained — no external `debug_*` script required.

- `CONFIG_PATH` — YAML config to use. Defaults to `configs/default.yaml`.
- `SEED` — RNG seed for reset() + policies.
- `CHECKPOINT_DIR` — directory containing `agent_<i>_best.pt` (or any of the `agent_<i>_*.pt` checkpoints saved by `train.py`). Set to `None` to fall back to untrained agents (random initialisation).
- `CHECKPOINT_PATTERN` — filename pattern to look for inside `CHECKPOINT_DIR` (e.g. `agent_{i}_best.pt`, `agent_{i}_final.pt`).
- `N_BOTS_OVERRIDE` / `N_LEARNING_OVERRIDE` — set to `None` to use what's in the config, or override (e.g. `N_BOTS_OVERRIDE=0` to remove bots from the view).
- `DETERMINISTIC` — if `True`, policies act on the mean (no exploration noise).
- `INSPECT_YEAR` — 1-based year to deep-dive (set to `None` to deep-dive the last simulated year).
- `INSPECT_AGENT` — agent index for the single-company walk-through.
- `EXPORT_DIR` — where to write the optional CSV exports (created if missing).


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# --- locate project root --------------------------------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "configs").exists() and (parent / "src").exists():
            PROJECT_ROOT = parent
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- USER CONTROLS --------------------------------------------------------
CONFIG_PATH         = PROJECT_ROOT / "configs" / "default.yaml"
SEED                = 42

# Set to a checkpoint directory (e.g. results/checkpoints_default_s42/) to load
# a trained policy and inspect "the final episode of a long run". Leave None to
# use freshly initialised PPO networks (useful for sanity-checking the env only).
CHECKPOINT_DIR      = None
CHECKPOINT_PATTERN  = "agent_{i}_best.pt"  # try "agent_{i}_final.pt" if _best is missing

# Override participant counts (None = use config values).
N_LEARNING_OVERRIDE = None
N_BOTS_OVERRIDE     = None

DETERMINISTIC       = True

# Year/agent of interest. INSPECT_YEAR is 1-based; None = last simulated year.
INSPECT_YEAR        = None
INSPECT_AGENT       = 0

# Export controls.
EXPORT_CSV          = False
EXPORT_DIR          = PROJECT_ROOT / "results" / "episode_viewer"

# Display controls (how wide to print pandas tables).
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


## 2. Load config & build environment

Loads the YAML, applies optional overrides, builds an `ETSEnvironment`, and (if `CHECKPOINT_DIR` is set) loads PPO actors via the same `build_agents` factory used by `scripts/train.py` and `scripts/evaluate.py`.


In [ ]:
from src.environment.ets_environment import ETSEnvironment
from scripts.train import build_agents

with open(CONFIG_PATH) as fh:
    cfg = yaml.safe_load(fh)

# Apply optional overrides BEFORE constructing the environment so all
# downstream sizing (obs/action dims, bot arrays, etc.) is consistent.
if N_LEARNING_OVERRIDE is not None:
    cfg["companies"]["n_agents"] = int(N_LEARNING_OVERRIDE)
if N_BOTS_OVERRIDE is not None:
    cfg["companies"]["n_bot_agents"] = int(N_BOTS_OVERRIDE)

env = ETSEnvironment(cfg, seed=SEED)
N_LEARNING = int(cfg["companies"]["n_agents"])
N_BOTS     = int(cfg["companies"]["n_bot_agents"])
N_TOTAL    = env.n_total
N_YEARS    = int(cfg["simulation"]["n_years"])
TECH_NAMES = list(cfg["technologies"]["names"])
BUILDABLE_TECH_NAMES = [TECH_NAMES[i] for i, b in enumerate(cfg["technologies"]["is_buildable"]) if b]

# Build PPO agents using the same factory as scripts/train.py and scripts/evaluate.py.
# When CHECKPOINT_DIR is supplied we load the trained weights; otherwise the agents
# stay at their random initialisation, which gives a clean "untrained baseline" view
# of the environment. Bots, when present, are driven by the env's own heuristic
# policy inside step_auction / step_secondary, so this notebook only needs to
# produce actions for the LEARNING agents.
agents = build_agents(env, cfg, SEED)
policy_mode = "untrained_ppo"
missing = []
if CHECKPOINT_DIR is not None:
    ckpt_root = Path(CHECKPOINT_DIR)
    if not ckpt_root.is_absolute():
        ckpt_root = PROJECT_ROOT / ckpt_root
    if not ckpt_root.exists():
        raise FileNotFoundError(f"CHECKPOINT_DIR not found: {ckpt_root}")
    for i, ag in enumerate(agents):
        ckpt_path = ckpt_root / CHECKPOINT_PATTERN.format(i=i)
        if ckpt_path.exists():
            ag.load(str(ckpt_path))
        else:
            missing.append(i)
    if missing:
        print(f"WARNING: missing checkpoints for agents {missing} — they keep random init.")
    policy_mode = f"checkpoint:{ckpt_root.name}"

print(f"Project root        : {PROJECT_ROOT}")
print(f"Config              : {CONFIG_PATH}")
print(f"Years simulated     : {N_YEARS}")
print(f"Learning agents     : {N_LEARNING}    Bots: {N_BOTS}    Total: {N_TOTAL}")
print(f"Policy mode         : {policy_mode}")
print(f"Deterministic actions: {DETERMINISTIC}")


## 3. Roll out one episode and capture everything

Helper functions:

- `participant_label(i)` — `A0..A{N_LEARNING-1}` for learners, `B0..B{N_BOTS-1}` for bots.
- `select_phase1/2_actions(obs)` — drives the **learning** agents using the actors built above (trained when a checkpoint was loaded; random init otherwise). Bot actions are produced inside the environment\'s `step_*` methods.

Each year we record:

1. The **pre-step snapshot** (annual budget, budget already spent, treasury, loan, bank, mix, queue size).
2. The **raw actions** (auction 6-D, secondary 2-D) — what the policy actually emitted, before any environment-side clipping/gating.
3. The full **`year_log`** dict returned by `step_secondary` (this is the same dict used by `train.py` for diagnostics).
4. The **post-step snapshot** of the same per-company state.


In [ ]:
def participant_label(i: int) -> str:
    return f"A{i}" if i < N_LEARNING else f"B{i - N_LEARNING}"

def participant_kind(i: int) -> str:
    return "learning" if i < N_LEARNING else "bot"

def select_phase1_actions(obs1):
    actions = np.zeros((N_LEARNING, 6), dtype=np.float32)
    for i in range(N_LEARNING):
        a, _, _ = agents[i].select_auction_action(obs1[i], deterministic=DETERMINISTIC)
        actions[i] = a
    return actions

def select_phase2_actions(obs2):
    actions = np.zeros((N_LEARNING, 2), dtype=np.float32)
    for i in range(N_LEARNING):
        a, _, _ = agents[i].select_secondary_action(obs2[i], deterministic=DETERMINISTIC)
        actions[i] = a
    return actions

def snapshot_companies():
    rows = []
    for i, c in enumerate(env.companies):
        rows.append({
            "agent": i,
            "label": participant_label(i),
            "kind": participant_kind(i),
            "annual_budget": float(c.annual_budget),
            "budget_spent": float(c.budget_spent_this_year),
            "budget_avail": float(c.annual_budget - c.budget_spent_this_year),
            "treasury": float(c._treasury_reserve),
            "treasury_drawn": float(c._treasury_drawn_this_year),
            "loan_outstanding": float(c._loan_outstanding),
            "bank_holdings": float(env.holdings[i]),
            "carry_forward": float(c._carry_forward),
            "green_frac": float(c.green_frac),
            "fossil_frac": float(c.fossil_frac),
            "queue_size": int(len(c._construction_queue)),
            **{f"mix_{TECH_NAMES[t]}": float(c.mix[t]) for t in range(len(TECH_NAMES))},
        })
    return pd.DataFrame(rows)

# ---- Reset & roll out -----------------------------------------------------
obs1, _ = env.reset(seed=SEED)
year_logs        = []   # one dict per simulated year
pre_snapshots    = []   # company snapshots BEFORE step_auction
post_snapshots   = []   # company snapshots AFTER step_secondary
raw_phase1_acts  = []   # raw learning-agent phase-1 actions
raw_phase2_acts  = []   # raw learning-agent phase-2 actions
total_rewards    = np.zeros(N_LEARNING)

for year in range(N_YEARS):
    pre_snapshots.append(snapshot_companies().assign(year=year + 1))

    a1 = select_phase1_actions(obs1)
    raw_phase1_acts.append(a1.copy())
    obs2, _ = env.step_auction(a1)

    a2 = select_phase2_actions(obs2)
    raw_phase2_acts.append(a2.copy())
    obs1, rewards, terminated, _, info = env.step_secondary(a2)
    total_rewards += rewards

    log = info["year_log"]
    log["_year_1based"] = year + 1
    year_logs.append(log)
    post_snapshots.append(snapshot_companies().assign(year=year + 1))

    if terminated:
        break

YEARS_DONE = len(year_logs)
print(f"Simulated {YEARS_DONE}/{N_YEARS} years.")
print("Total learning-agent rewards (raw):")
display(pd.DataFrame({
    "agent": [participant_label(i) for i in range(N_LEARNING)],
    "total_reward": total_rewards,
}))

# Resolve INSPECT_YEAR / INSPECT_AGENT now that we know YEARS_DONE.
if INSPECT_YEAR is None:
    INSPECT_YEAR = YEARS_DONE
if not (1 <= INSPECT_YEAR <= YEARS_DONE):
    raise ValueError(f"INSPECT_YEAR must be in [1, {YEARS_DONE}], got {INSPECT_YEAR}.")
if not (0 <= INSPECT_AGENT < N_TOTAL):
    raise ValueError(f"INSPECT_AGENT must be in [0, {N_TOTAL - 1}], got {INSPECT_AGENT}.")
print(f"Year deep-dive: INSPECT_YEAR={INSPECT_YEAR}    Single-company walk-through: INSPECT_AGENT={INSPECT_AGENT} ({participant_label(INSPECT_AGENT)})")


## 4. Episode summary — Market perspective

One row per simulated year. Use this to spot regime shifts (price hitting reserve, MSR firing, defaults, etc.) at a glance, before drilling into a specific year.


In [ ]:
def _g(d, key, default=np.nan):
    v = d.get(key, default)
    return v if v is not None else default

market_rows = []
for log in year_logs:
    stats = log.get("auction_stats", {}) or {}
    market_rows.append({
        "year"                 : log["_year_1based"],
        "cap_mt"               : float(_g(log, "cap")),
        "tnac_mt"              : float(_g(log, "tnac")),
        "msr_reserve_mt"       : float(_g(log, "msr_reserve", 0.0)),
        "msr_withhold_mt"      : float(_g(log, "msr_withhold_this_year", 0.0)),
        "msr_release_mt"       : float(_g(log, "msr_release_this_year", 0.0)),
        "msr_total_cancelled"  : float(_g(log, "msr_total_cancelled", 0.0)),
        "auction_volume_mt"    : float(_g(log, "auction_volume")),
        "rollover_in_mt"       : float(_g(log, "unsold_rollover_in", 0.0)),
        "rollover_out_mt"      : float(_g(log, "unsold_rollover_out", 0.0)),
        "default_rolled_in_mt" : float(_g(log, "defaulted_volume_rolled_in", 0.0)),
        "reserve_eur"          : float(_g(log, "effective_reserve")),
        "anchor_eur"           : float(_g(log, "anchor_t")) if log.get("anchor_t") is not None else np.nan,
        "auction_clearing_eur" : float(_g(log, "clearing_price")),
        "secondary_clearing_eur": float(_g(log, "secondary_clearing")),
        "auction_demand_mt"    : float(stats.get("total_demand", np.nan)),
        "auction_allocated_mt" : float(stats.get("total_allocated", np.nan)),
        "auction_failed"       : bool(stats.get("auction_failed", False)),
        "auction_defaults"     : int(stats.get("defaults", 0)),
        "secondary_volume_mt"  : float(_g(log, "secondary_volume", 0.0)),
        "phantom_active"       : bool(_g(log, "phantom_active", False)),
        "phantom_bid_eur"      : float(_g(log, "phantom_bid_price", np.nan)),
        "phantom_bid_qty_mt"   : float(_g(log, "phantom_bid_qty", np.nan)),
        "emissions_total_mt"   : float(np.sum(_g(log, "emissions", []))),
        "shortfalls_total_mt"  : float(np.sum(_g(log, "shortfalls", []))),
        "penalties_total_meur" : float(np.sum(_g(log, "penalties", []))),
        "invest_costs_total_meur": float(np.sum(_g(log, "invest_costs", []))),
        "trade_costs_total_meur" : float(np.sum(_g(log, "trade_costs", []))),
        "auction_payments_total_meur": float(np.sum(_g(log, "payments", []))),
        "inflation_factor"     : float(_g(log, "inflation_factor", 1.0)),
        "mean_reward_learning" : float(np.mean(_g(log, "rewards", [np.nan])[:N_LEARNING])) if log.get("rewards") else np.nan,
    })
market_summary_df = pd.DataFrame(market_rows)
print("Market summary (one row per year)")
display(market_summary_df)


## 5. Episode summary — Per-company perspective

One row per (year, agent). Useful for tracking each firm's bank, mix, transactions, and reward across the whole episode.


In [ ]:
def _arr(log, key, n=N_TOTAL, default=0.0):
    v = log.get(key)
    if v is None:
        return [default] * n
    return list(v)

per_company_rows = []
for log in year_logs:
    yr = log["_year_1based"]
    holdings   = _arr(log, "holdings")
    emissions  = _arr(log, "emissions")
    payments   = _arr(log, "payments")
    trade_q    = _arr(log, "trade_qtys")
    trade_c    = _arr(log, "trade_costs")
    invest_c   = _arr(log, "invest_costs")
    mac_c      = _arr(log, "mac_costs")
    coll_c     = _arr(log, "collateral_costs")
    pen        = _arr(log, "penalties")
    short      = _arr(log, "shortfalls")
    rewards    = _arr(log, "rewards", n=N_LEARNING, default=np.nan)
    bid_p      = _arr(log, "bid_prices", default=np.nan)
    bid_q      = _arr(log, "bid_quantities", default=np.nan)
    invest_f   = _arr(log, "invest_fracs", default=0.0)
    tech_choice= _arr(log, "invest_tech_choices", default=-1)
    green_fracs= _arr(log, "green_fracs")
    delta_g    = _arr(log, "delta_greens")
    queue_sz   = _arr(log, "queue_sizes", default=0)
    treas      = _arr(log, "treasury_reserves")
    treas_d    = _arr(log, "treasury_drawn")
    loans      = _arr(log, "loan_outstanding")
    cf_in      = _arr(log, "old_carry_forward")
    cancels    = _arr(log, "cancellations")
    em_shocks  = _arr(log, "emission_shocks", n=N_LEARNING, default=np.nan)
    allocs     = _arr(log, "allocations")
    for i in range(N_TOTAL):
        per_company_rows.append({
            "year": yr,
            "agent": i,
            "label": participant_label(i),
            "kind": participant_kind(i),
            "bank_pre_compliance": np.nan,  # filled below from snapshots
            "bank_post_compliance": float(holdings[i]),
            "allocation_mt"      : float(allocs[i]),
            "auction_payment_meur": float(payments[i]),
            "bid_price_eur"      : float(bid_p[i]) if i < len(bid_p) else np.nan,
            "bid_qty_mt"         : float(bid_q[i]) if i < len(bid_q) else np.nan,
            "trade_qty_mt"       : float(trade_q[i]),
            "trade_cost_meur"    : float(trade_c[i]),
            "invest_frac"        : float(invest_f[i]) if i < len(invest_f) else 0.0,
            "tech_choice"        : (BUILDABLE_TECH_NAMES[int(tech_choice[i])]
                                    if 0 <= int(tech_choice[i]) < len(BUILDABLE_TECH_NAMES) else ""),
            "invest_cost_meur"   : float(invest_c[i]),
            "mac_cost_meur"      : float(mac_c[i]),
            "collateral_cost_meur": float(coll_c[i]),
            "penalty_meur"       : float(pen[i]),
            "shortfall_mt"       : float(short[i]),
            "carry_forward_in_mt": float(cf_in[i]),
            "cancellation_mt"    : float(cancels[i]),
            "emissions_mt"       : float(emissions[i]),
            "emission_shock"     : float(em_shocks[i]) if i < len(em_shocks) else np.nan,
            "green_frac"         : float(green_fracs[i]),
            "delta_green"        : float(delta_g[i]),
            "queue_size"         : int(queue_sz[i]),
            "treasury"           : float(treas[i]),
            "treasury_drawn"     : float(treas_d[i]),
            "loan_outstanding"   : float(loans[i]),
            "reward"             : float(rewards[i]) if i < N_LEARNING else np.nan,
        })

per_company_df = pd.DataFrame(per_company_rows)
print("Per-company yearly summary (long format)")
display(per_company_df.head(min(len(per_company_df), N_TOTAL * 3)))
print(f"... {len(per_company_df)} rows total. Filter by `agent`/`label` or pivot as needed.")


## 6. Year deep-dive — Market panel

Everything the auction & secondary market did in `INSPECT_YEAR`, in one place. Use this together with sections 7–11 below for the company-side view.


In [ ]:
year_log = year_logs[INSPECT_YEAR - 1]
stats = year_log.get("auction_stats", {}) or {}

market_panel = pd.Series({
    "year"                       : INSPECT_YEAR,
    "cap_mt"                     : year_log.get("cap"),
    "tnac_mt"                    : year_log.get("tnac"),
    "msr_reserve_mt"             : year_log.get("msr_reserve"),
    "msr_withhold_this_year_mt"  : year_log.get("msr_withhold_this_year"),
    "msr_release_this_year_mt"   : year_log.get("msr_release_this_year"),
    "msr_total_cancelled_mt"     : year_log.get("msr_total_cancelled"),
    "auction_volume_mt"          : year_log.get("auction_volume"),
    "unsold_rollover_in_mt"      : year_log.get("unsold_rollover_in"),
    "unsold_rollover_out_mt"     : year_log.get("unsold_rollover_out"),
    "defaulted_volume_rolled_in_mt": year_log.get("defaulted_volume_rolled_in"),
    "effective_reserve_eur"      : year_log.get("effective_reserve"),
    "fundamental_anchor_eur"     : year_log.get("anchor_t"),
    "auction_clearing_eur"       : year_log.get("clearing_price"),
    "secondary_clearing_eur"     : year_log.get("secondary_clearing"),
    "secondary_volume_mt"        : year_log.get("secondary_volume"),
    "auction_total_demand_mt"    : stats.get("total_demand"),
    "auction_total_allocated_mt" : stats.get("total_allocated"),
    "auction_failed"             : stats.get("auction_failed"),
    "auction_defaults"           : stats.get("defaults"),
    "auction_default_agents"     : stats.get("defaults_agents"),
    "phantom_active"             : year_log.get("phantom_active"),
    "phantom_bid_price_eur"      : year_log.get("phantom_bid_price"),
    "phantom_bid_qty_mt"         : year_log.get("phantom_bid_qty"),
    "inflation_factor"           : year_log.get("inflation_factor"),
    "inflation_rate"             : year_log.get("inflation_rate"),
    "marginal_emission_factor"   : year_log.get("marginal_ef_used"),
}, name=f"market@year{INSPECT_YEAR}")
print(f"Market panel for year {INSPECT_YEAR}")
display(market_panel.to_frame())

liq = year_log.get("liquidity_pool", {})
if liq:
    print("Liquidity pool")
    display(pd.Series(liq, name="liquidity_pool").to_frame())


## 7. Year deep-dive — Bidding ledger

Per-participant view of what each company *intended* (raw policy action, qty multiplier, estimate of need) versus what was *executed* (clipped/gated bid price & qty, allocation, payment, default flag).


In [ ]:
y0 = INSPECT_YEAR - 1
a1_raw = raw_phase1_acts[y0]   # shape (N_LEARNING, 6)

bid_rows = []
default_set = set(year_log.get("auction_stats", {}).get("defaults_agents", []) or [])
for i in range(N_TOTAL):
    raw_bid_p   = float(a1_raw[i, 0]) if i < N_LEARNING else np.nan
    raw_qty_m   = float(a1_raw[i, 1]) if i < N_LEARNING else np.nan
    raw_inv_f   = float(a1_raw[i, 2]) if i < N_LEARNING else np.nan
    raw_tech    = (BUILDABLE_TECH_NAMES[int(np.argmax(a1_raw[i, 3:6]))]
                   if i < N_LEARNING else "")
    bid_rows.append({
        "agent"             : i,
        "label"             : participant_label(i),
        "kind"              : participant_kind(i),
        "raw_bid_price_eur" : raw_bid_p,
        "raw_qty_mult"      : raw_qty_m,
        "raw_invest_frac"   : raw_inv_f,
        "raw_tech_choice"   : raw_tech,
        "estimate_need_mt"  : float(year_log.get("estimate_needs", [np.nan]*N_TOTAL)[i]),
        "qty_mult_executed" : float(year_log.get("bid_qty_multipliers", [np.nan]*N_TOTAL)[i]),
        "bid_price_executed_eur": float(year_log.get("bid_prices", [np.nan]*N_TOTAL)[i])
                                   if year_log.get("bid_prices") else np.nan,
        "bid_qty_executed_mt": float(year_log.get("bid_quantities", [np.nan]*N_TOTAL)[i]),
        "bid_to_reserve_ratio": float(year_log.get("bid_to_reserve_ratio", [np.nan]*N_TOTAL)[i])
                                  if year_log.get("bid_to_reserve_ratio") else np.nan,
        "bid_coverage"      : float(year_log.get("bid_coverages", [np.nan]*N_TOTAL)[i]),
        "allocation_mt"     : float(year_log.get("allocations", [np.nan]*N_TOTAL)[i]),
        "auction_payment_meur": float(year_log.get("payments", [np.nan]*N_TOTAL)[i]),
        "collateral_cost_meur": float(year_log.get("collateral_costs", [np.nan]*N_TOTAL)[i]),
        "defaulted"         : (i in default_set),
    })
bidding_df = pd.DataFrame(bid_rows)
print(f"Auction bidding ledger — year {INSPECT_YEAR}")
display(bidding_df)


## 8. Year deep-dive — Secondary market & compliance ledger

Every secondary-market action (raw vs. gated quantity, executed price) and every compliance outcome (emissions vs. surrendered allowances, shortfall, penalty, carry-forward, cancellation) for `INSPECT_YEAR`.


In [ ]:
a2_raw = raw_phase2_acts[y0]

sec_rows = []
for i in range(N_TOTAL):
    raw_p = float(a2_raw[i, 0]) if i < N_LEARNING else np.nan
    raw_q = float(a2_raw[i, 1]) if i < N_LEARNING else np.nan
    sec_rows.append({
        "agent"            : i,
        "label"            : participant_label(i),
        "kind"             : participant_kind(i),
        "raw_sec_price_eur": raw_p,
        "raw_sec_qty_mt"   : raw_q,
        "exec_sec_price_eur": float(year_log.get("sec_price_mults", [np.nan]*N_TOTAL)[i]),
        "raw_intent_qty_mt": float(year_log.get("raw_secondary_qtys", [np.nan]*N_TOTAL)[i])
                             if year_log.get("raw_secondary_qtys") else np.nan,
        "gated_qty_mt"     : float(year_log.get("gated_secondary_qtys", [np.nan]*N_TOTAL)[i])
                             if year_log.get("gated_secondary_qtys") else np.nan,
        "executed_qty_mt"  : float(year_log.get("trade_qtys", [np.nan]*N_TOTAL)[i]),
        "trade_cost_meur"  : float(year_log.get("trade_costs", [np.nan]*N_TOTAL)[i]),
        "side"             : ({-1: "sell", 0: "hold", 1: "buy"}
                               .get(int(year_log.get("sec_action_sides", [0]*N_TOTAL)[i]), "?")),
    })
secondary_df = pd.DataFrame(sec_rows)
print(f"Secondary market ledger — year {INSPECT_YEAR}")
display(secondary_df)

comp_rows = []
em_shocks = year_log.get("emission_shocks", [])
for i in range(N_TOTAL):
    comp_rows.append({
        "agent"             : i,
        "label"             : participant_label(i),
        "kind"              : participant_kind(i),
        "carry_forward_in_mt": float(year_log.get("old_carry_forward", [0.0]*N_TOTAL)[i]),
        "allocation_mt"     : float(year_log.get("allocations", [0.0]*N_TOTAL)[i]),
        "secondary_qty_mt"  : float(year_log.get("trade_qtys", [0.0]*N_TOTAL)[i]),
        "emissions_mt"      : float(year_log.get("emissions", [0.0]*N_TOTAL)[i]),
        "emission_shock"    : float(em_shocks[i]) if i < len(em_shocks) else np.nan,
        "post_holdings_mt"  : float(year_log.get("holdings", [0.0]*N_TOTAL)[i]),
        "shortfall_mt"      : float(year_log.get("shortfalls", [0.0]*N_TOTAL)[i]),
        "penalty_meur"      : float(year_log.get("penalties", [0.0]*N_TOTAL)[i]),
        "cancellation_mt"   : float(year_log.get("cancellations", [0.0]*N_TOTAL)[i]),
    })
compliance_df = pd.DataFrame(comp_rows)
print(f"Compliance ledger — year {INSPECT_YEAR}")
display(compliance_df)


## 9. Year deep-dive — Investment ledger

What each company invested in this year (executed `invest_frac`, tech choice, capex), how much MAC abatement they bought, treasury draws, and outstanding loan changes. The end-of-year queue size shows whether new capacity is in the pipeline.


In [ ]:
inv_rows = []
pre_df  = pre_snapshots[y0].set_index("agent")
post_df = post_snapshots[y0].set_index("agent")
for i in range(N_TOTAL):
    inv_rows.append({
        "agent"               : i,
        "label"               : participant_label(i),
        "kind"                : participant_kind(i),
        "invest_frac"         : float(year_log.get("invest_fracs", [0.0]*N_TOTAL)[i])
                                 if i < N_LEARNING and year_log.get("invest_fracs") else 0.0,
        "tech_choice"         : (BUILDABLE_TECH_NAMES[int(year_log.get("invest_tech_choices",
                                                                       [-1]*N_TOTAL)[i])]
                                  if (i < N_LEARNING
                                      and 0 <= int(year_log.get("invest_tech_choices",
                                                                [-1]*N_TOTAL)[i]) < len(BUILDABLE_TECH_NAMES))
                                  else ""),
        "invest_cost_meur"    : float(year_log.get("invest_costs", [0.0]*N_TOTAL)[i]),
        "mac_reduction_mt"    : float(year_log.get("mac_reductions", [0.0]*N_TOTAL)[i]),
        "mac_cost_meur"       : float(year_log.get("mac_costs", [0.0]*N_TOTAL)[i]),
        "queue_size_pre"      : int(pre_df.loc[i, "queue_size"]),
        "queue_size_post"     : int(post_df.loc[i, "queue_size"]),
        "green_frac_pre"      : float(pre_df.loc[i, "green_frac"]),
        "green_frac_post"     : float(post_df.loc[i, "green_frac"]),
        "delta_green"         : float(year_log.get("delta_greens", [0.0]*N_TOTAL)[i]),
        "treasury_pre"        : float(pre_df.loc[i, "treasury"]),
        "treasury_post"       : float(post_df.loc[i, "treasury"]),
        "treasury_drawn_year" : float(year_log.get("treasury_drawn", [0.0]*N_TOTAL)[i]),
        "loan_pre"            : float(pre_df.loc[i, "loan_outstanding"]),
        "loan_post"           : float(post_df.loc[i, "loan_outstanding"]),
        "effective_capex_throughput": float(year_log.get("effective_capex_throughput",
                                                          [np.nan]*N_TOTAL)[i]),
    })
investment_df = pd.DataFrame(inv_rows)
print(f"Investment ledger — year {INSPECT_YEAR}")
display(investment_df)


## 10. Year deep-dive — Budget ledger

Decomposes each participant's spending in `INSPECT_YEAR` into its components: auction payment, secondary trade cost, investment cost, MAC cost, collateral cost, penalty. Shows budget at start vs. end and the resulting headroom.


In [ ]:
budget_rows = []
for i in range(N_TOTAL):
    pay   = float(year_log.get("payments", [0.0]*N_TOTAL)[i])
    trd   = float(year_log.get("trade_costs", [0.0]*N_TOTAL)[i])
    inv   = float(year_log.get("invest_costs", [0.0]*N_TOTAL)[i])
    mac   = float(year_log.get("mac_costs", [0.0]*N_TOTAL)[i])
    coll  = float(year_log.get("collateral_costs", [0.0]*N_TOTAL)[i])
    pen   = float(year_log.get("penalties", [0.0]*N_TOTAL)[i])
    budget_rows.append({
        "agent"                  : i,
        "label"                  : participant_label(i),
        "kind"                   : participant_kind(i),
        "annual_budget_meur"     : float(pre_df.loc[i, "annual_budget"]),
        "budget_spent_pre_meur"  : float(pre_df.loc[i, "budget_spent"]),
        "auction_payment_meur"   : pay,
        "trade_cost_meur"        : trd,
        "invest_cost_meur"       : inv,
        "mac_cost_meur"          : mac,
        "collateral_cost_meur"   : coll,
        "penalty_meur"           : pen,
        "total_outflow_meur"     : pay + trd + inv + mac + coll + pen,
        "budget_spent_post_meur" : float(post_df.loc[i, "budget_spent"]),
        "budget_avail_post_meur" : float(post_df.loc[i, "budget_avail"]),
    })
budget_df = pd.DataFrame(budget_rows)
print(f"Budget ledger — year {INSPECT_YEAR}")
display(budget_df)


## 11. Year deep-dive — Reward channels & per-agent diagnostics

`reward_channels` is the per-agent decomposition of the raw reward (auction cost, baseline cost, capital_norm, ESG terms, banking signal, …). `per_agent_diag` carries WTP, bid heuristics and coverage diagnostics — useful for sanity-checking whether the joint budget gate or compliance affordability is binding.


In [ ]:
rc = year_log.get("reward_channels", {}) or {}
if rc:
    rc_rows = []
    for i, ch in rc.items():
        row = {"agent": int(i), "label": participant_label(int(i))}
        row.update({k: float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v
                    for k, v in ch.items()})
        rc_rows.append(row)
    reward_channels_df = pd.DataFrame(rc_rows).sort_values("agent").reset_index(drop=True)
    print(f"Reward channels — year {INSPECT_YEAR}")
    display(reward_channels_df)
else:
    reward_channels_df = pd.DataFrame()
    print("No reward_channels recorded for this year.")

pad = year_log.get("per_agent_diag", {}) or {}
if pad:
    diag_rows = []
    for i, d in pad.items():
        row = {"agent": int(i), "label": participant_label(int(i))}
        row.update(d)
        diag_rows.append(row)
    per_agent_diag_df = pd.DataFrame(diag_rows).sort_values("agent").reset_index(drop=True)
    print(f"Per-agent auction diagnostics — year {INSPECT_YEAR}")
    display(per_agent_diag_df)
else:
    per_agent_diag_df = pd.DataFrame()
    print("No per_agent_diag recorded for this year.")


## 12. Single-company walk-through

Trajectory of `INSPECT_AGENT` across all simulated years: budget, transactions, investments, mix, holdings, reward. This is the column you'd typically copy/paste into a research note when explaining one company's strategy.


In [ ]:
walk_df = (
    per_company_df[per_company_df["agent"] == INSPECT_AGENT]
    .copy()
    .reset_index(drop=True)
)
mix_history = []
for snap in post_snapshots:
    s = snap[snap["agent"] == INSPECT_AGENT].iloc[0]
    mix_history.append({
        "year": int(s["year"]),
        **{f"mix_{t}": float(s[f"mix_{t}"]) for t in TECH_NAMES},
    })
mix_history_df = pd.DataFrame(mix_history)

print(f"Trajectory — agent {INSPECT_AGENT} ({participant_label(INSPECT_AGENT)})")
display(walk_df)

print(f"Tech mix evolution — agent {INSPECT_AGENT}")
display(mix_history_df)


## 13. Quick plots

A handful of charts that surface the most common research-question lenses:
1. Auction clearing vs. fundamental anchor vs. effective reserve.
2. TNAC vs. cap (over/under-allocation regime).
3. Per-agent post-compliance bank.
4. Per-agent emissions vs. allocation.

Disable / extend as needed — all the underlying data is already in the dataframes above.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ax = axes[0, 0]
ax.plot(market_summary_df["year"], market_summary_df["auction_clearing_eur"], "-o", label="auction clearing")
ax.plot(market_summary_df["year"], market_summary_df["secondary_clearing_eur"], "-s", label="secondary clearing")
ax.plot(market_summary_df["year"], market_summary_df["anchor_eur"], "--", label="fundamental anchor")
ax.plot(market_summary_df["year"], market_summary_df["reserve_eur"], ":", label="effective reserve")
ax.set_title("Prices (EUR/t)"); ax.set_xlabel("year"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.plot(market_summary_df["year"], market_summary_df["cap_mt"], "-o", label="cap")
ax.plot(market_summary_df["year"], market_summary_df["tnac_mt"], "-s", label="TNAC")
ax.plot(market_summary_df["year"], market_summary_df["msr_reserve_mt"], "--", label="MSR reserve")
ax.set_title("Cap / TNAC / MSR (Mt)"); ax.set_xlabel("year"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 0]
for i in range(N_TOTAL):
    sub = per_company_df[per_company_df["agent"] == i]
    ax.plot(sub["year"], sub["bank_post_compliance"], label=participant_label(i))
ax.set_title("Post-compliance bank by participant (Mt)"); ax.set_xlabel("year")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)

ax = axes[1, 1]
for i in range(N_TOTAL):
    sub = per_company_df[per_company_df["agent"] == i]
    ax.plot(sub["year"], sub["emissions_mt"] - sub["allocation_mt"], label=participant_label(i))
ax.axhline(0, color="k", lw=0.5)
ax.set_title("Emissions − allocation (Mt) — positive = short")
ax.set_xlabel("year"); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 14. Optional CSV export

Set `EXPORT_CSV = True` at the top of the notebook to dump every dataframe above into `EXPORT_DIR/`. Filenames are tagged with `INSPECT_YEAR` and `INSPECT_AGENT` so multiple invocations can coexist.


In [ ]:
if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    tag = f"y{INSPECT_YEAR:02d}_a{INSPECT_AGENT:02d}_seed{SEED}"
    market_summary_df.to_csv(EXPORT_DIR / f"market_summary_{tag}.csv", index=False)
    per_company_df.to_csv(EXPORT_DIR / f"per_company_yearly_{tag}.csv", index=False)
    bidding_df.to_csv(EXPORT_DIR / f"bidding_year{INSPECT_YEAR:02d}.csv", index=False)
    secondary_df.to_csv(EXPORT_DIR / f"secondary_year{INSPECT_YEAR:02d}.csv", index=False)
    compliance_df.to_csv(EXPORT_DIR / f"compliance_year{INSPECT_YEAR:02d}.csv", index=False)
    investment_df.to_csv(EXPORT_DIR / f"investment_year{INSPECT_YEAR:02d}.csv", index=False)
    budget_df.to_csv(EXPORT_DIR / f"budget_year{INSPECT_YEAR:02d}.csv", index=False)
    if not reward_channels_df.empty:
        reward_channels_df.to_csv(EXPORT_DIR / f"reward_channels_year{INSPECT_YEAR:02d}.csv", index=False)
    if not per_agent_diag_df.empty:
        per_agent_diag_df.to_csv(EXPORT_DIR / f"per_agent_diag_year{INSPECT_YEAR:02d}.csv", index=False)
    walk_df.to_csv(EXPORT_DIR / f"agent{INSPECT_AGENT:02d}_walkthrough.csv", index=False)
    mix_history_df.to_csv(EXPORT_DIR / f"agent{INSPECT_AGENT:02d}_mix_history.csv", index=False)
    print(f"Wrote CSV exports to {EXPORT_DIR}")
else:
    print("EXPORT_CSV is False — set it to True at the top of the notebook to dump CSVs.")


## 15. Raw `year_log` access

Every dataframe above is derived from `year_logs[INSPECT_YEAR - 1]`. The raw dict is kept around for ad-hoc deep dives — print its keys here, and pull whatever else you need (e.g. `opponent_snapshots`, `terminal_*_values`, `rewards_base`, `rewards_shaping`, etc.).


In [ ]:
print(f"year_log keys (year {INSPECT_YEAR}):")
display(pd.DataFrame({"key": sorted(year_log.keys())}))

# Example ad-hoc pull:
# rewards_base    = year_log["rewards_base"]
# rewards_shaping = year_log["rewards_shaping"]
# opponent_snap   = year_log["opponent_snapshots"]
